# Arabic Root Extraction — QLoRA Fine-Tuning v2 (Unsloth) — Kaggle

Fine-tune **Gemma 3 1B IT** with QLoRA using **Unsloth** for 2× faster training
and **`train_on_responses_only`** to focus loss exclusively on root tokens.

**Goal**: Beat the best rule-based tool (ISRI Stemmer, 68.6% exact match) on our
16,277-entry golden test set from 6 lexicographic sources.

## Key Changes from v1 → v2

| Parameter | v1 | v2 | Why |
|-----------|-----|-----|-----|
| Framework | bitsandbytes + PEFT + SFTTrainer | **Unsloth FastModel** | 2× faster, 58% less VRAM |
| `train_on_responses_only` | No | **Yes** | Focus loss on root tokens only |
| Epochs | 3 | **10** | v1 loss still falling at step 200 |
| LR scheduler | `constant` | **`cosine`** | Better for longer training |
| Optimizer | `adamw_torch_fused` | **`adamw_8bit`** | 50% less optimizer memory |
| `lora_dropout` | 0.05 | **0** | Unsloth-optimized |
| Packing | True | **False** | Required for response-only masking |

## Prerequisites (Kaggle)

1. Upload `train.json` and `test.json` as a **Kaggle Dataset** (e.g., `arabic-root-extraction-data`)
2. Accept the Gemma license at [huggingface.co/google/gemma-3-1b-it](https://huggingface.co/google/gemma-3-1b-it)
3. Add these as **Kaggle Secrets** (Add-ons → Secrets in sidebar):
   - `HF_TOKEN` — Hugging Face access token
   - `WANDB_API_KEY` — Weights & Biases API key ([wandb.ai/authorize](https://wandb.ai/authorize))
4. Enable **GPU T4 × 2** accelerator (Settings → Accelerator)
5. Enable **Internet** access (Settings → Internet → On)

## Baselines (from v1)

| Tool | Type | Exact Match |
|------|------|-------------|
| ISRI Stemmer (NLTK) | rule_based | 68.6% |
| Tashaphyne | rule_based | 62.1% |
| **Gemma-3-1B v1 (QLoRA)** | **neural_finetuned** | **60.8%** |
| SinaTools (ALMA) | hybrid | 40.8% |
| Gemma-3-1B-IT (zero-shot) | zero_shot_llm | 26.7% |

## Cell 1 — Install Dependencies (Unsloth)

In [1]:
%%capture
import os, re

# Kaggle and non-Colab environments use the standard pip install
if "COLAB_" in "".join(os.environ.keys()):
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
else:
    !pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install wandb

## Cell 2 — Auth & Paths (Kaggle)

On Kaggle, secrets are accessed via `kaggle_secrets.UserSecretsClient`.
Data uploaded as a Kaggle Dataset is mounted read-only at `/kaggle/input/<dataset-name>/`.
All writable output goes to `/kaggle/working/` (persists as notebook output, max 20 GB).

In [2]:
import os, json, gc, glob, re, random, time
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from huggingface_hub import login
import wandb

# ── Authenticate (Kaggle Secrets) ────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

HF_TOKEN = secrets.get_secret("HF_TOKEN")
WANDB_API_KEY = secrets.get_secret("WANDB_API_KEY")

login(HF_TOKEN)
wandb.login(key=WANDB_API_KEY)

# ── Configure paths (Kaggle) ─────────────────────────────────────────────
# IMPORTANT: Update KAGGLE_DATASET_NAME to match your uploaded dataset name.
# Your dataset should contain train.json and test.json at the top level.
KAGGLE_DATASET_NAME = "arabic-root-extraction-data"  # <-- change this to your dataset slug

DATA_DIR    = f"/kaggle/input/{KAGGLE_DATASET_NAME}"
OUTPUT_DIR  = "/kaggle/working/outputs-v2"

# ── Configure Hub repo ───────────────────────────────────────────────────
HF_USERNAME     = "SalahAbdoNLP"
HUB_REPO_ID     = f"{HF_USERNAME}/gemma-3-1b-arabic-root-v2"
MERGED_HUB_REPO = f"{HUB_REPO_ID}-merged"

# ── Configure wandb ─────────────────────────────────────────────────────
WANDB_PROJECT   = "arabic-root-extraction"
WANDB_RUN_ID    = "gemma-root-qlora-v2-kaggle"

# ── Locate data files ───────────────────────────────────────────────────
# Try common layouts: flat or nested under data/splits/
train_path = None
for candidate in [
    f"{DATA_DIR}/train.json",
    f"{DATA_DIR}/data/splits/train.json",
    f"{DATA_DIR}/splits/train.json",
]:
    if Path(candidate).exists():
        train_path = candidate
        break

test_path = None
for candidate in [
    f"{DATA_DIR}/test.json",
    f"{DATA_DIR}/data/splits/test.json",
    f"{DATA_DIR}/splits/test.json",
]:
    if Path(candidate).exists():
        test_path = candidate
        break

assert train_path, (
    f"train.json not found under {DATA_DIR}. "
    f"Upload your dataset and update KAGGLE_DATASET_NAME. "
    f"Contents: {list(Path(DATA_DIR).rglob('*.json'))[:10] if Path(DATA_DIR).exists() else 'DIR NOT FOUND'}"
)
assert test_path, f"test.json not found under {DATA_DIR}."

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Train data:  {train_path}")
print(f"Test data:   {test_path}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Hub repo:    {HUB_REPO_ID}")

ConnectionError: Connection error trying to communicate with service.

## Cell 3 — Load Model with Unsloth FastModel

Unsloth's `FastModel` replaces manual bitsandbytes quantization + PEFT setup.
It handles 4-bit quantization, optimized attention kernels, and gradient
checkpointing internally — 2× faster and 58% less VRAM than vanilla HuggingFace.

We use `max_seq_length=128` because our data is very short (~30–40 tokens with chat template).

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it",
    max_seq_length = 128,
    load_in_4bit = True,
    full_finetuning = False,
)

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Cell 4 — LoRA Adapters

Unsloth's `get_peft_model` replaces manual `LoraConfig` + `get_peft_model` from PEFT.
Key differences from v1:
- `lora_dropout=0` — Unsloth's optimized kernels don't benefit from dropout
- `r=16, lora_alpha=16` — same as v1 (1:1 ratio recommended by Unsloth)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({trainable/total:.2%})")

## Cell 5 — Load & Format Dataset

Same dataset and split as v1, but formatted using Unsloth's helpers:
- `get_chat_template(tokenizer, "gemma-3")` — applies Gemma 3 chat format
- `standardize_data_formats(dataset)` — normalizes conversation structure
- `removeprefix('<bos>')` — the processor adds `<bos>` during training, so we strip duplicates

We use the `conversations` key (ShareGPT format) instead of v1's `messages` key.

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_data_formats
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template = "gemma-3")

# ── Load raw entries ────────────────────────────────────────────────────
with open(train_path, encoding="utf-8") as f:
    train_entries = json.load(f)
print(f"Loaded {len(train_entries):,} training entries")


# ── Root-based split (no leakage) ───────────────────────────────────────
def split_by_root(entries, val_ratio=0.10, seed=42):
    """Split entries into train/val by root to prevent leakage."""
    roots = list(set(e["root_gold"] for e in entries))
    random.seed(seed)
    random.shuffle(roots)
    n_val = max(1, int(len(roots) * val_ratio))
    val_roots = set(roots[:n_val])
    train = [e for e in entries if e["root_gold"] not in val_roots]
    val   = [e for e in entries if e["root_gold"] in val_roots]
    return train, val

train_split, val_split = split_by_root(train_entries, val_ratio=0.10, seed=42)
print(f"Train: {len(train_split):,}  |  Val: {len(val_split):,}")


# ── Format as conversations (ShareGPT style) ──────────────────────────
def to_conversation(entry):
    return {"conversations": [
        {"role": "user",      "content": f"\u0627\u0633\u062a\u062e\u0631\u062c \u062c\u0630\u0631: {entry['word']}"},
        {"role": "assistant", "content": entry["root_gold"]},
    ]}

train_ds = Dataset.from_list([to_conversation(e) for e in train_split])
val_ds   = Dataset.from_list([to_conversation(e) for e in val_split])

train_ds = standardize_data_formats(train_ds)
val_ds   = standardize_data_formats(val_ds)


# ── Apply chat template ────────────────────────────────────────────────
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        ).removeprefix('<bos>')
        for convo in convos
    ]
    return {"text": texts}

train_ds = train_ds.map(formatting_prompts_func, batched=True)
val_ds   = val_ds.map(formatting_prompts_func, batched=True)

print(f"\nSample formatted text:")
print(train_ds[0]["text"])

## Cell 6 — Training Setup + Response-Only Masking

The **key technique** in v2: `train_on_responses_only` masks the instruction tokens
so loss is computed ONLY on the root output. In v1, the model wasted gradient signal
learning to predict the repeated instruction `استخرج جذر:` prefix.

We verify masking by printing the labels — masked tokens show as spaces.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 4,   # effective batch = 64
        num_train_epochs = 10,
        warmup_steps = 50,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        max_grad_norm = 0.3,
        logging_steps = 25,
        eval_strategy = "epoch",
        save_strategy = "steps",
        save_steps = 500,
        save_total_limit = 3,
        seed = 42,
        report_to = "wandb",
        push_to_hub = True,
        hub_model_id = HUB_REPO_ID,
        hub_strategy = "checkpoint",
        hub_private_repo = True,
        output_dir = OUTPUT_DIR,
    ),
)

# KEY TECHNIQUE: mask instruction tokens — loss only on root output
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

# ── Verify masking ──────────────────────────────────────────────────────
print("Full input tokens:")
print(tokenizer.decode(trainer.train_dataset[0]["input_ids"]))
print("\nMasked labels (spaces = masked out, only root gets gradient):")
labels = trainer.train_dataset[0]["labels"]
print(tokenizer.decode(
    [tokenizer.pad_token_id if x == -100 else x for x in labels]
).replace(tokenizer.pad_token, " "))

## Cell 7 — Train

10 epochs with cosine LR scheduler.

**Kaggle note:** `/kaggle/working/` is volatile — if the session dies, local checkpoints
are lost. The `push_to_hub=True` setting saves checkpoints to HF Hub as insurance.
To resume from a Hub checkpoint, see the fallback logic below.

In [ ]:
# ── Initialize wandb ────────────────────────────────────────────────────
wandb.init(
    project=WANDB_PROJECT,
    name="gemma-3-1b-it-qlora-v2-kaggle",
    id=WANDB_RUN_ID,
    resume="allow",
    tags=["qlora", "gemma", "arabic", "root-extraction", "v2", "unsloth", "kaggle"],
    config={
        "base_model": "unsloth/gemma-3-1b-it",
        "framework": "unsloth",
        "platform": "kaggle",
        "lora_r": 16,
        "lora_alpha": 16,
        "epochs": 10,
        "lr_scheduler": "cosine",
        "train_on_responses_only": True,
        "dataset_size": len(train_split),
        "val_size": len(val_split),
    },
)
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

# ── Memory before training ─────────────────────────────────────────────
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved before training.")

# ── Auto-resume from local checkpoint (if session is still alive) ──────
# On Kaggle, local checkpoints only persist within a single session.
# For cross-session resume, pull from HF Hub (see commented block below).
checkpoints = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")))
if checkpoints:
    latest_ckpt = max(checkpoints, key=os.path.getmtime)
    print(f"\nResuming from local checkpoint: {latest_ckpt}")
    trainer_stats = trainer.train(resume_from_checkpoint=latest_ckpt)
else:
    # ── Optional: resume from HF Hub checkpoint (uncomment to enable) ──
    # from huggingface_hub import snapshot_download
    # hub_ckpt = snapshot_download(
    #     repo_id=HUB_REPO_ID,
    #     token=HF_TOKEN,
    #     local_dir=f"{OUTPUT_DIR}/hub-checkpoint",
    # )
    # print(f"Resuming from Hub checkpoint: {hub_ckpt}")
    # trainer_stats = trainer.train(resume_from_checkpoint=hub_ckpt)
    print("\nStarting training from scratch...")
    trainer_stats = trainer.train()

# ── Memory after training ──────────────────────────────────────────────
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_training = round(used_memory - start_gpu_memory, 3)
print(f"\n{trainer_stats.metrics['train_runtime']:.0f} seconds used for training.")
print(f"{trainer_stats.metrics['train_runtime']/60:.1f} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_training} GB.")
print(f"Peak reserved memory % of max = {round(used_memory / max_memory * 100, 1)}%.")

## Cell 8 — Sanity Check

Quick inference on 10 test words before full evaluation.

In [ ]:
ARABIC_ROOT_RE = re.compile(r"[\u0621-\u064A]+")

def parse_root(text):
    """Extract the first plausible Arabic root from model output."""
    text = text.strip()
    matches = ARABIC_ROOT_RE.findall(text)
    if not matches:
        return ""
    for m in matches:
        if 2 <= len(m) <= 5:
            return m
    return matches[0] if matches else ""


def predict_root(word, model, tokenizer):
    """Predict root for a single word."""
    messages = [{"role": "user", "content": f"\u0627\u0633\u062a\u062e\u0631\u062c \u062c\u0630\u0631: {word}"}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=16, do_sample=False,
            temperature=None, top_p=None,
        )
    new_tokens = outputs[0][prompt_len:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return parse_root(raw), raw


# Load test samples
with open(test_path, encoding="utf-8") as f:
    test_entries = json.load(f)

print(f"Loaded {len(test_entries):,} test entries")
print("\n" + "=" * 50)
print("Sanity check: 10 random test words")
print("=" * 50)

random.seed(42)
sample_indices = random.sample(range(len(test_entries)), 10)
correct = 0
for idx in sample_indices:
    entry = test_entries[idx]
    pred, raw = predict_root(entry["word"], model, tokenizer)
    match = pred == entry["root_gold"]
    correct += int(match)
    tag = "OK" if match else "XX"
    print(f"  [{tag}] {entry['word']:15s} -> {pred:6s} (gold: {entry['root_gold']})")
print(f"\nSanity check: {correct}/10 correct")

## Cell 9 — Merge, Export & Push to Hub

Merge LoRA adapters into the base model and save as float16 for deployment.
Unsloth's `save_pretrained_merged` handles this in one step.

In [ ]:
# ── Save LoRA adapter ──────────────────────────────────────────────────
ADAPTER_PATH = f"{OUTPUT_DIR}/lora-adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter saved to {ADAPTER_PATH}")

# ── Merge + save float16 ───────────────────────────────────────────────
MERGED_PATH = f"{OUTPUT_DIR}/merged"
print(f"Merging adapter into base model...")
model.save_pretrained_merged(MERGED_PATH, tokenizer)
size_gb = sum(f.stat().st_size for f in Path(MERGED_PATH).rglob("*") if f.is_file()) / 1e9
print(f"Merged model saved to {MERGED_PATH} ({size_gb:.1f} GB)")

# ── Push merged model to Hub ───────────────────────────────────────────
print(f"\nPushing merged model to Hub: {MERGED_HUB_REPO}...")
model.push_to_hub_merged(
    MERGED_HUB_REPO, tokenizer,
    token = HF_TOKEN,
    private = True,
)
print("Pushed to Hub.")

# ── Optional: GGUF export (uncomment to enable) ─────────────────────────
# GGUF_PATH = f"{OUTPUT_DIR}/gguf"
# model.save_pretrained_gguf(GGUF_PATH, tokenizer, quantization_method="Q8_0")
# print(f"GGUF saved to {GGUF_PATH}")

## Cell 10 — Full Evaluation on Test Set

Evaluate the merged model on all 16,277 test entries with the same normalization
and metrics used in v1 and our local `evaluate.py` framework.

We reload the merged float16 model for a fair comparison with v1.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Arabic normalization (matches evaluate.py exactly) ──────────────────
DIACRITICS_RE = re.compile("[\u064B-\u065F\u0670]")

def normalize_for_comparison(root):
    if not root:
        return ""
    root = DIACRITICS_RE.sub("", root)
    root = re.sub("[\u0623\u0625\u0622]", "\u0627", root)  # أإآ → ا
    root = root.replace("\u0649", "\u064A")                 # ى → ي
    root = root.replace("\u0640", "")                       # tatweel
    root = root.replace(" ", "").strip()
    return root

def edit_distance(s1, s2):
    if len(s1) < len(s2):
        return edit_distance(s2, s1)
    if not s2:
        return len(s1)
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(
                prev[j + 1] + 1,
                curr[j] + 1,
                prev[j] + (0 if c1 == c2 else 1),
            ))
        prev = curr
    return prev[-1]


# ── Free training model, load merged model ────────────────────────────
del model, trainer
torch.cuda.empty_cache()
gc.collect()

print("Loading merged model for evaluation...")
eval_model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
eval_tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH)
eval_model.eval()
print(f"Model loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


# ── Batch prediction ──────────────────────────────────────────────────
def predict_roots_batch(words, model, tokenizer, batch_size=64):
    """Predict roots for a list of words using batched inference."""
    all_predictions = []
    all_raw = []
    for i in range(0, len(words), batch_size):
        batch_words = words[i:i + batch_size]
        prompts = []
        for w in batch_words:
            messages = [{"role": "user", "content": f"\u0627\u0633\u062a\u062e\u0631\u062c \u062c\u0630\u0631: {w}"}]
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            prompts.append(prompt)
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True,
            max_length=64,
        ).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=16, do_sample=False,
                temperature=None, top_p=None,
            )
        prompt_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            new_tokens = output[prompt_len:]
            raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
            pred = parse_root(raw)
            all_predictions.append(pred)
            all_raw.append(raw)
    return all_predictions, all_raw


# ── Run full evaluation ────────────────────────────────────────────────
print(f"\nEvaluating on {len(test_entries):,} test entries...")
print("=" * 60)

words = [e["word"] for e in test_entries]
t0 = time.time()
predictions, raw_outputs = predict_roots_batch(
    words, eval_model, eval_tokenizer, batch_size=64
)
elapsed = time.time() - t0
print(f"Inference done: {elapsed:.1f}s ({len(words) / elapsed:.0f} words/sec)")


# ── Compute metrics ────────────────────────────────────────────────────
results = []
for i, entry in enumerate(test_entries):
    gold = normalize_for_comparison(entry["root_gold"])
    alt_roots = [normalize_for_comparison(r) for r in entry.get("alternative_roots", [])]
    valid_roots = {gold} | set(alt_roots)
    pred = normalize_for_comparison(predictions[i])
    is_covered = bool(pred)
    is_exact = pred in valid_roots if pred else False
    ed = edit_distance(pred, gold) if pred else len(gold)
    results.append({
        "word": entry["word"],
        "gold": entry["root_gold"],
        "predicted": predictions[i],
        "raw_output": raw_outputs[i],
        "correct": is_exact,
        "edit_distance": ed,
        "covered": is_covered,
        "source": entry.get("source", ""),
        "root_length": entry.get("root_length", 0),
        "has_weak_radical": entry.get("has_weak_radical", False),
        "word_complexity": entry.get("word_complexity", ""),
        "era": entry.get("era", ""),
    })


def compute_metrics(subset):
    n = len(subset)
    if n == 0:
        return {"count": 0}
    n_correct = sum(1 for r in subset if r["correct"])
    n_covered = sum(1 for r in subset if r["covered"])
    total_ed  = sum(r["edit_distance"] for r in subset)
    return {
        "count": n,
        "exact_match": round(n_correct / n, 4),
        "coverage": round(n_covered / n, 4),
        "accuracy_on_covered": round(n_correct / max(n_covered, 1), 4),
        "mean_edit_distance": round(total_ed / n, 2),
    }

overall = compute_metrics(results)

by_root_length = {}
for rl in sorted(set(r["root_length"] for r in results)):
    by_root_length[rl] = compute_metrics([r for r in results if r["root_length"] == rl])

by_complexity = {}
for comp in ["bare_root", "simple_derived", "complex_derived"]:
    by_complexity[comp] = compute_metrics([r for r in results if r["word_complexity"] == comp])

by_era = {}
for era in ["modern", "classical"]:
    by_era[era] = compute_metrics([r for r in results if r["era"] == era])

by_weak = {
    "strong": compute_metrics([r for r in results if not r["has_weak_radical"]]),
    "weak":   compute_metrics([r for r in results if r["has_weak_radical"]]),
}

errors = [r for r in results if not r["correct"] and r["covered"]]

eval_results = {
    "overall": overall,
    "by_root_length": by_root_length,
    "by_complexity": by_complexity,
    "by_era": by_era,
    "by_weak_radical": by_weak,
    "timing": {"total_seconds": round(elapsed, 2), "words_per_second": round(len(words) / elapsed, 1)},
    "error_samples": errors[:50],
}

print(f"\n{'=' * 60}")
print(f"RESULTS: Gemma-3-1B v2 (Unsloth QLoRA + train_on_responses_only)")
print(f"{'=' * 60}")
print(f"  Exact match:     {overall['exact_match']:.1%}")
print(f"  Coverage:        {overall['coverage']:.1%}")
print(f"  Acc@covered:     {overall['accuracy_on_covered']:.1%}")
print(f"  Mean edit dist:  {overall['mean_edit_distance']}")
print(f"  Speed:           {eval_results['timing']['words_per_second']:,.0f} words/sec")
print(f"  Total entries:   {overall['count']:,}")

## Cell 11 — Results Tables & Comparison

In [ ]:
# ── Comparison table ────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("COMPARISON WITH BASELINES")
print("=" * 70)

baselines = [
    ("ISRI Stemmer (NLTK)",         "rule_based",     0.686),
    ("Tashaphyne",                  "rule_based",     0.621),
    ("Gemma-3-1B v1 (QLoRA)",       "neural_finetuned", 0.608),
    ("SinaTools (ALMA)",            "hybrid",         0.408),
    ("Qalsadi",                     "hybrid",         0.387),
    ("CAMeL Tools (CALIMA)",        "lexicon",        0.301),
    ("Gemma-3-1B-IT (zero-shot)",   "zero_shot_llm",  0.267),
]

our_result = overall["exact_match"]
our_row = ("Gemma-3-1B v2 (Unsloth)", "neural_finetuned", our_result)

all_rows = baselines + [our_row]
all_rows.sort(key=lambda x: x[2], reverse=True)

print(f"\n{'Tool':<35s} {'Type':<20s} {'Exact Match':>12s}")
print("-" * 70)
for name, ttype, score in all_rows:
    marker = " <<<" if "v2" in name else ""
    print(f"{name:<35s} {ttype:<20s} {score:>11.1%}{marker}")

beat_isri = our_result > 0.686
improvement_v1 = (our_result - 0.608) / 0.608 * 100
improvement_zs = (our_result - 0.267) / 0.267 * 100
print(f"\n{'=' * 70}")
print(f"Beat ISRI baseline (68.6%): {'YES' if beat_isri else 'NO'}")
print(f"Improvement over v1 (60.8%): {'+' if improvement_v1 >= 0 else ''}{improvement_v1:.1f}%")
print(f"Improvement over zero-shot (26.7%): +{improvement_zs:.0f}%")


# ── Stratified breakdowns ─────────────────────────────────────────────
print(f"\n\n{'=' * 60}")
print("BY ROOT LENGTH (v1 \u2192 v2)")
print("=" * 60)
v1_by_rl = {2: 0.423, 3: 0.607, 4: 0.714, 5: 0.467}
print(f"{'Root Length':<15s} {'Count':>8s} {'v1':>8s} {'v2':>8s} {'Delta':>8s}")
print("-" * 50)
for rl in sorted(by_root_length.keys()):
    m = by_root_length[rl]
    if m["count"] > 0:
        v1_val = v1_by_rl.get(rl, 0)
        delta = m["exact_match"] - v1_val
        print(f"{rl}-letter{'':<8s} {m['count']:>8,} {v1_val:>7.1%} {m['exact_match']:>7.1%} {'+' if delta >= 0 else ''}{delta:>6.1%}")

print(f"\n\n{'=' * 60}")
print("BY WORD COMPLEXITY")
print("=" * 60)
print(f"{'Complexity':<20s} {'Count':>8s} {'Exact Match':>12s} {'Coverage':>10s}")
print("-" * 55)
for comp in ["bare_root", "simple_derived", "complex_derived"]:
    m = by_complexity[comp]
    if m["count"] > 0:
        print(f"{comp:<20s} {m['count']:>8,} {m['exact_match']:>11.1%} {m['coverage']:>9.1%}")

print(f"\n\n{'=' * 60}")
print("BY ERA")
print("=" * 60)
print(f"{'Era':<15s} {'Count':>8s} {'Exact Match':>12s} {'Acc@Covered':>12s}")
print("-" * 50)
for era in ["modern", "classical"]:
    m = by_era[era]
    if m["count"] > 0:
        print(f"{era:<15s} {m['count']:>8,} {m['exact_match']:>11.1%} {m['accuracy_on_covered']:>11.1%}")

print(f"\n\n{'=' * 60}")
print("BY WEAK RADICAL")
print("=" * 60)
print(f"{'Type':<15s} {'Count':>8s} {'Exact Match':>12s} {'Coverage':>10s}")
print("-" * 50)
for wtype in ["strong", "weak"]:
    m = by_weak[wtype]
    if m["count"] > 0:
        print(f"{wtype:<15s} {m['count']:>8,} {m['exact_match']:>11.1%} {m['coverage']:>9.1%}")


# ── Error samples ──────────────────────────────────────────────────────
print(f"\n\n{'=' * 60}")
print("ERROR SAMPLES (first 20)")
print("=" * 60)
print(f"{'Word':<15s} {'Gold':>8s} {'Predicted':>10s} {'Edit Dist':>10s}")
print("-" * 50)
for e in eval_results["error_samples"][:20]:
    print(f"{e['word']:<15s} {e['gold']:>8s} {e['predicted'] or '\u2014':>10s} {e['edit_distance']:>10d}")

## Cell 12 — Save Results & Log to wandb

In [ ]:
results_dir = f"{OUTPUT_DIR}/eval_results"
os.makedirs(results_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ── Save JSON results ──────────────────────────────────────────────────
results_data = {
    "run_id": timestamp,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "platform": "kaggle",
    "model": "unsloth/gemma-3-1b-it (QLoRA v2 + Unsloth + train_on_responses_only)",
    "adapter_path": ADAPTER_PATH,
    "merged_path": MERGED_PATH,
    "hub_repo": HUB_REPO_ID,
    "merged_hub_repo": MERGED_HUB_REPO,
    "dataset_size": len(test_entries),
    "training_size": len(train_split),
    "validation_size": len(val_split),
    "config": {
        "framework": "unsloth",
        "lora_r": 16,
        "lora_alpha": 16,
        "epochs": 10,
        "batch_size": 16,
        "grad_accum": 4,
        "lr": 2e-4,
        "lr_scheduler": "cosine",
        "optimizer": "adamw_8bit",
        "train_on_responses_only": True,
        "packing": False,
    },
    "evaluation": eval_results,
}

json_path = f"{results_dir}/results_{timestamp}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results_data, f, ensure_ascii=False, indent=2)

# ── Save Markdown report ───────────────────────────────────────────────
report_lines = [
    "# Arabic Root Extraction \u2014 QLoRA v2 Results (Unsloth, Kaggle)",
    "",
    f"**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    f"**Platform:** Kaggle",
    f"**Model:** unsloth/gemma-3-1b-it (QLoRA v2, r=16, 10 epochs, cosine LR)",
    f"**Key technique:** train_on_responses_only + Unsloth FastModel",
    f"**Training set:** {len(train_split):,} entries (90% of {len(train_entries):,})",
    f"**Test set:** {len(test_entries):,} entries",
    f"**Hub repo:** [{MERGED_HUB_REPO}](https://huggingface.co/{MERGED_HUB_REPO})",
    "",
    "---",
    "",
    "## Summary",
    "",
    "| Metric | Value |",
    "|--------|-------|",
    f"| Exact match | {overall['exact_match']:.1%} |",
    f"| Coverage | {overall['coverage']:.1%} |",
    f"| Acc@covered | {overall['accuracy_on_covered']:.1%} |",
    f"| Mean edit distance | {overall['mean_edit_distance']} |",
    f"| Speed | {eval_results['timing']['words_per_second']:,.0f} words/sec |",
    "",
    "---",
    "",
    "## Comparison",
    "",
    "| Tool | Type | Exact Match |",
    "|------|------|-------------|",
]
for name, ttype, score in all_rows:
    marker = " **" if "v2" in name else ""
    report_lines.append(f"| {marker}{name}{marker} | {ttype} | {score:.1%} |")

report_lines += [
    "",
    "## By Root Length",
    "",
    "| Root Length | Count | Exact Match |",
    "|------------|-------|-------------|",
]
for rl in sorted(by_root_length.keys()):
    m = by_root_length[rl]
    if m["count"] > 0:
        report_lines.append(f"| {rl}-letter | {m['count']:,} | {m['exact_match']:.1%} |")

report_path = f"{results_dir}/report_{timestamp}.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print(f"Results saved to /kaggle/working/:")
print(f"  {json_path}")
print(f"  {report_path}")


# ── Log to wandb ───────────────────────────────────────────────────────
wandb.log({
    "eval/exact_match": overall["exact_match"],
    "eval/coverage": overall["coverage"],
    "eval/accuracy_on_covered": overall["accuracy_on_covered"],
    "eval/mean_edit_distance": overall["mean_edit_distance"],
    "eval/speed_words_per_sec": eval_results["timing"]["words_per_second"],
})

for rl, m in by_root_length.items():
    if m["count"] > 0:
        wandb.log({f"eval/root_length_{rl}/exact_match": m["exact_match"]})
for era, m in by_era.items():
    if m["count"] > 0:
        wandb.log({f"eval/era_{era}/exact_match": m["exact_match"]})

error_table = wandb.Table(
    columns=["word", "gold", "predicted", "edit_distance"],
    data=[[e["word"], e["gold"], e["predicted"] or "\u2014", e["edit_distance"]]
          for e in eval_results["error_samples"][:50]],
)
wandb.log({"eval/error_samples": error_table})

# Upload merged model as artifact
print("\nUploading merged model as wandb artifact...")
artifact = wandb.Artifact(
    name="gemma-3-1b-arabic-root-v2-merged",
    type="model",
    description="Gemma 3 1B IT v2: Unsloth + train_on_responses_only + 10 epochs cosine (Kaggle)",
    metadata={"exact_match": overall["exact_match"], "hub_repo": MERGED_HUB_REPO},
)
artifact.add_dir(MERGED_PATH)
wandb.log_artifact(artifact)

wandb.finish()

print(f"\nAll done!")
print(f"  Output:  {OUTPUT_DIR}")
print(f"  Hub:     https://huggingface.co/{MERGED_HUB_REPO}")
print(f"  wandb:   https://wandb.ai/{WANDB_PROJECT}/runs/{WANDB_RUN_ID}")